<a href="https://colab.research.google.com/github/tiwariaxay/PDF-RAG-CHATBOT/blob/chatbot/PDF_RAG_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install -q transformers sentence-transformers faiss-cpu pypdf torch accelerate

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded PDF: {pdf_path}")

In [ ]:
from pypdf import PdfReader

reader = PdfReader(pdf_path)
raw_text = ""

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        raw_text += text + "\n"

print(f"Extracted characters: {len(raw_text)}")
print(raw_text[:1500])

In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

chunks = chunk_text(raw_text)
print(f"Total chunks created: {len(chunks)}")

In [ ]:
for i, chunk in enumerate(chunks[:5]):
    print(f"--- Chunk {i+1} ---")
    print(chunk[:800])
    print()

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embedder.encode(chunks, show_progress_bar=True)

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index ready")

total_vectors = index.ntotal

# to display the vectors
# for i in range(total_vectors):
#     vector = index.reconstruct(i)
#     print(i, vector)

In [ ]:
def retrieve_context(query, top_k=5):
    query_embedding = embedder.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)
    return [chunks[i] for i in indices[0]]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/long-t5-tglobal-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model

In [ ]:
def build_prompt(context_chunks, question):
    context = "\n\n".join(context_chunks)
    prompt = f"""
Answer the question strictly using the context below.
If the answer is not present, respond with "I don't know."

Context:
{context}

Question:
{question}

Answer:
"""
    return prompt

In [ ]:
def generate_answer(prompt, max_new_tokens=200):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
def chat():
    print("PDF RAG Chatbot (type 'exit' to stop)")
    while True:
        question = input("\nUser: ")
        if question.lower() == "exit":
            break

        retrieved = retrieve_context(question, top_k=5)
        prompt = build_prompt(retrieved, question)
        answer = generate_answer(prompt)
        print("\nAssistant:", answer.split("Answer:")[-1].strip())

chat()